# Step 3: Retry Middleware 실습 -- State 기반 vs 미들웨어 비교

Step 3에서 `RetryState` + 조건부 엣지로 구현한 재시도 로직을,
LangChain 미들웨어 패턴으로 얼마나 간결하게 대체할 수 있는지 비교합니다.

| 실습 목표 |
|---|
| 1. State 기반 재시도 그래프의 구조를 복습합니다 |
| 2. 미들웨어 패턴으로 동일 동작을 구현합니다 |
| 3. 두 접근법의 트레이드오프를 이해합니다 |

> **참고**: `ToolRetryMiddleware`는 LangChain 버전에 따라 API가 다를 수 있습니다.
> 이 노트북은 **개념 비교**에 초점을 두며, 실행 시 버전에 맞게 import를 조정하시기 바랍니다.

In [ ]:
# === Imports & LLM 설정 ===
import os
from typing import Literal, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

# 환경변수에서 API 키를 읽습니다 (하드코딩 금지)
assert os.environ.get("OPENROUTER_API_KEY"), (
    "OPENROUTER_API_KEY 환경변수가 설정되지 않았습니다. "
    "export OPENROUTER_API_KEY='sk-or-...' 로 설정하세요."
)

# OpenRouter 게이트웨이를 통해 LLM 사용
llm = ChatOpenAI(
    model="google/gemini-3-flash-preview",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)
print("LLM 설정 완료")

### 방식 1: State 기반 재시도 (Step 3 복습)

Step 3에서 직접 구현한 방식입니다.
`RetryState`에 `retry_count`를 넣고, 조건부 엣지(`should_retry`)로 분기했습니다.

```
START --> LLM --> tools_condition --> Tool (handle_tool_with_retry)
                                         |
                                         v
                                    should_retry?
                                   /            \
                            재시도 가능        최대 도달
                               |                 |
                              LLM               END
```

이 방식은 **그래프 흐름에서 재시도가 명시적으로 보인다**는 장점이 있습니다.
반면 노드 3개, 엣지 조건 함수 2개 등 코드량이 많습니다.

In [ ]:
# 불안정한 외부 API를 모사하는 Tool
# 처음 2번은 실패하고, 3번째부터 성공합니다.

@tool
def unreliable_api_call(query: str) -> str:
    """불안정한 외부 API를 호출합니다. 데모를 위해 처음 2번은 실패합니다."""
    attempts = getattr(unreliable_api_call, "_attempts", 0) + 1
    unreliable_api_call._attempts = attempts

    if attempts < 3:
        raise Exception("API 타임아웃: 서버 응답 없음")
    return f"API 결과: '{query}'에 대한 응답입니다. (시도 {attempts}회)"

print("unreliable_api_call 도구 정의 완료")

In [ ]:
# === State 기반 재시도 그래프 구현 ===

# 1) RetryState: MessagesState에 retry_count, max_retries 필드 추가
class RetryState(MessagesState):
    retry_count: int
    max_retries: int


# 2) LLM 노드: 시스템 프롬프트와 함께 Tool 호출 결정
def call_model_with_retry(state: RetryState) -> dict:
    system = SystemMessage(
        content=(
            "당신은 메일 관리 비서입니다. "
            "문제를 해결할 때 unreliable_api_call 도구를 반드시 사용하세요. "
            "도구 실패 메시지를 받으면 재시도하고, 성공하면 최종 답변을 짧게 정리하세요."
        )
    )
    retry_llm = llm.bind_tools([unreliable_api_call])
    response = retry_llm.invoke([system] + state["messages"])
    if response.tool_calls:
        names = ", ".join(tc["name"] for tc in response.tool_calls)
        print(f"  [LLM] Tool 호출 결정: {names}")
    else:
        print(f"  [LLM] 최종 답변: {response.content[:80]}")
    return {"messages": [response]}


# 3) Tool 노드: 실행 시 에러가 나면 retry_count 증가
def handle_tool_with_retry(state: RetryState) -> dict:
    last_msg = state["messages"][-1]
    results = []
    for tool_call in last_msg.tool_calls:
        try:
            result = unreliable_api_call.invoke(tool_call["args"])
            results.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
            print(f"  [Tool] 성공: {str(result)[:60]}")
        except Exception as e:
            retry_count = state.get("retry_count", 0) + 1
            error_msg = f"에러 (시도 {retry_count}/{state.get('max_retries', 3)}): {e}"
            results.append(ToolMessage(content=error_msg, tool_call_id=tool_call["id"]))
            print(f"  [Tool] {error_msg}")
            return {"messages": results, "retry_count": retry_count}
    return {"messages": results, "retry_count": 0}


# 4) 조건부 엣지: 재시도 여부 판단
def should_retry(state: RetryState) -> Literal["llm", "__end__"]:
    retry_count = state.get("retry_count", 0)
    max_retries = state.get("max_retries", 3)
    if retry_count > 0 and retry_count < max_retries:
        print(f"  [분기] 재시도 {retry_count}/{max_retries} -> LLM으로 되돌아감")
        return "llm"
    elif retry_count >= max_retries:
        print(f"  [분기] 최대 재시도 도달 ({max_retries}회) -> END")
        return END
    print(f"  [분기] 성공 -> LLM에게 결과 전달")
    return "llm"


# 5) 그래프 조립
retry_builder = StateGraph(RetryState)
retry_builder.add_node("llm", call_model_with_retry)
retry_builder.add_node("tools", handle_tool_with_retry)
retry_builder.add_edge(START, "llm")
retry_builder.add_conditional_edges("llm", tools_condition)
retry_builder.add_conditional_edges("tools", should_retry, ["llm", END])
retry_graph = retry_builder.compile()

print("State 기반 재시도 그래프 컴파일 완료")
print("노드: llm, tools / 조건 엣지: tools_condition, should_retry")

In [ ]:
# === State 기반 재시도 실행 ===
unreliable_api_call._attempts = 0  # 카운터 초기화

print("[실행] State 기반 재시도 데모")
print("=" * 50)

result = retry_graph.invoke({
    "messages": [HumanMessage(content="외부 API를 사용해 서버 상태를 확인해줘")],
    "retry_count": 0,
    "max_retries": 3,
})

print("=" * 50)
print(f"최종 응답: {result['messages'][-1].content}")
print(f"Tool 호출 횟수: {unreliable_api_call._attempts}회")

### 방식 2: ToolRetryMiddleware 사용

LangChain은 `ToolRetryMiddleware`를 통해 Tool 호출 실패 시 자동 재시도를 지원합니다.
위에서 약 30줄로 구현한 재시도 로직(State 필드 추가 + 커스텀 Tool 노드 + 조건부 엣지)을
미들웨어 설정 몇 줄로 대체할 수 있습니다.

**핵심 차이**: State 기반 방식은 그래프 구조 자체에 재시도가 녹아 있지만,
미들웨어 방식은 Tool 실행 계층에서 **암묵적으로** 재시도가 일어납니다.

> **참고**: `ToolRetryMiddleware`는 LangChain 버전에 따라 API가 다를 수 있습니다.
> 아래 코드가 실행되지 않는 경우, 사용 중인 LangChain 버전의 공식 문서를 확인하시기 바랍니다.

In [ ]:
# === 방식 2: Middleware 기반 재시도 (빈칸 채우기) ===
#
# 아래 빈칸(___) 을 채워서 미들웨어 기반 재시도 에이전트를 완성하세요.
# 힌트는 주석에 있습니다.

from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware


# 불안정한 Tool 재정의 (방식 1과 독립적으로 카운터 관리)
@tool
def unstable_lookup(keyword: str) -> str:
    """불안정한 외부 조회 API를 모사합니다."""
    ATTEMPT["count"] += 1
    if ATTEMPT["count"] <= 2:
        raise RuntimeError("일시적 네트워크 오류")
    return f"조회 성공: '{keyword}' 관련 메일 2건"

ATTEMPT = {"count": 0}


# 빈칸을 채우세요
retry_middleware = ToolRetryMiddleware(
    max_retries=___,        # 힌트: 최대 재시도 횟수 (정수)
)

agent = create_agent(
    model=llm,
    tools=[unstable_lookup],
    system_prompt="도구를 사용해 서버 상태를 확인하세요.",
    middleware=[___],        # 힌트: 바로 위에서 만든 미들웨어 *인스턴스* 변수명
)

In [ ]:
# === Middleware 기반 재시도 실행 ===
ATTEMPT["count"] = 0  # 카운터 초기화

print("[실행] Middleware 기반 재시도 데모")
print("=" * 50)

result = agent.invoke({
    "messages": [{"role": "user", "content": "keyword='긴급' 메일을 조회해줘."}]
})

print("=" * 50)
print(f"Tool 호출 시도 횟수: {ATTEMPT['count']}회")
for i, msg in enumerate(result.get("messages", []), 1):
    msg_type = type(msg).__name__
    content = getattr(msg, "content", "")
    print(f"[{i}] {msg_type}: {content[:100]}")

### 비교 정리

| 항목 | State 기반 (방식 1) | 미들웨어 (방식 2) |
|------|-------------------|------------------|
| **코드량** | ~30줄 (State 정의 + 커스텀 Tool 노드 + 조건부 엣지) | ~5줄 (미들웨어 생성 + 설정) |
| **가시성** | 그래프 시각화에서 재시도 흐름이 명확히 보임 | 암묵적 -- 그래프에는 재시도가 보이지 않음 |
| **커스터마이징** | 자유로움 (재시도 조건, 백오프 전략, 로깅 등 직접 제어) | 미들웨어가 제공하는 설정 범위 내에서만 가능 |
| **디버깅** | State에 retry_count가 남아 있어 추적이 쉬움 | 미들웨어 내부 동작을 확인하려면 로깅 설정 필요 |
| **적합한 상황** | 재시도 로직이 복잡하거나, 실패 시 다른 경로로 분기해야 할 때 | 단순한 재시도가 필요할 때, 빠르게 프로토타이핑할 때 |

**결론**: 두 방식은 상호 배타적이지 않습니다.
- 프로토타입 단계에서는 미들웨어로 빠르게 구현하고,
- 프로덕션에서 세밀한 제어가 필요하면 State 기반으로 전환하는 것이 일반적인 패턴입니다.

---

## 정답

<details>
<summary><b>클릭하여 정답 확인</b></summary>

#### 방식 2 빈칸 정답

```python
retry_middleware = ToolRetryMiddleware(
    max_retries=3,
)

agent = create_agent(
    model=llm,
    tools=[unstable_lookup],
    system_prompt="도구를 사용해 서버 상태를 확인하세요.",
    middleware=[retry_middleware],
)
```

**해설:**
- `max_retries=3`: Step 3의 State 기반 구현에서 `max_retries` 필드에 넣었던 값과 동일합니다.
- `middleware=[retry_middleware]`: 미들웨어 리스트에 위에서 만든 객체를 전달합니다.
- `system_prompt=`: `create_agent()`의 파라미터명은 `prompt`가 아니라 **`system_prompt`** 입니다.

미들웨어가 내부적으로 실패를 감지하고 같은 Tool 호출을 최대 3회까지 자동 재시도합니다.

**State 기반과의 차이:**
- State 기반에서는 `RetryState` 정의 + `handle_tool_with_retry` 노드 + `should_retry` 조건부 엣지가 모두 필요했습니다.
- 미들웨어에서는 `ToolRetryMiddleware(max_retries=3)` 한 줄로 같은 동작을 얻습니다.
- 대신 미들웨어 방식은 그래프에 재시도 흐름이 보이지 않아, 디버깅 시 로깅 설정이 별도로 필요합니다.

</details>